In [36]:
import pandas as pd
import pymysql
import os
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

conn = pymysql.connect(
    host=os.environ.get('DB_HOST', 'localhost'),
    user=os.environ.get('DB_USER', 'root'),
    password=os.environ.get('DB_PASSWORD', 'root'),
    database='ads_db',
    charset='utf8mb4'
)
print("数据库连接成功")

数据库连接成功


In [ ]:
# 【口径】车间级指标用「加权」口径：SUM(actual_qty)/SUM(plan_qty)、
#        SUM(qualified_qty)/SUM(actual_qty)，与 dws_produce_day / 预警 / 大屏口径一致。
#        不要用 AVG(每行比率)（未加权均值）；也没必要把 25 万行明细拉进内存再 groupby，
#        这里直接把聚合下推到 SQL，只取回 6 行。
ws = pd.read_sql("""
    SELECT workshop_name,
           ROUND(SUM(actual_qty)    * 100.0 / NULLIF(SUM(plan_qty), 0),    2) AS capacity_achieved,
           ROUND(SUM(qualified_qty) * 100.0 / NULLIF(SUM(actual_qty), 0), 2) AS qualified_rate
    FROM ads_produce_monitor
    GROUP BY workshop_name
""", conn).set_index('workshop_name')

# 整体口径同样用加权（SUM/SUM），而不是"各车间均值再平均"
overall = pd.read_sql("""
    SELECT COUNT(*) AS rows_cnt,
           ROUND(SUM(actual_qty)    * 100.0 / NULLIF(SUM(plan_qty), 0),    2) AS overall_cap,
           ROUND(SUM(qualified_qty) * 100.0 / NULLIF(SUM(actual_qty), 0), 2) AS overall_qual
    FROM ads_produce_monitor
""", conn).iloc[0]
conn.close()

overall_cap = float(overall['overall_cap'])
overall_qual = float(overall['overall_qual'])
print(f"读取 {int(overall['rows_cnt'])} 条明细 → 聚合为 {len(ws)} 个车间（加权口径 SUM/SUM）")
print(f"整体产能达成率: {overall_cap:.2f}%　整体良品率: {overall_qual:.2f}%")
ws


In [ ]:
# ws 已在上一个 cell 按「加权口径」聚合好，这里只做展示
print(f"整体产能达成率: {overall_cap:.2f}%")
print(f"整体良品率: {overall_qual:.2f}%")
print("\n各车间（加权口径 SUM/SUM）:")
ws_ala = ws.copy()
ws_ala.rename(columns={
    "capacity_achieved": "产能达成率%",
    "qualified_rate": "良品率%",
}, inplace=True)
print(ws_ala.to_markdown(index=True, numalign="right", stralign="left"))


In [39]:
# 第 4 段：四象限分类（固定管理标准 80% / 95%）
# 判定规则：
#双优     : 产能达成率 ≥ 80% 且 良品率 ≥ 95%  → 标杆，保持并输出经验
#高产低质 : 产能达成率 ≥ 80% 但 良品率 < 95%  → 优先改善质量
#低产高质 : 产能达成率 < 80% 但 良品率 ≥ 95%  → 优先提升产能
#双低     : 产能达成率 < 80% 且 良品率 < 95%  → 产能与质量双线整改
CAP_STD, QUAL_STD = 80.0, 95.0


def classify_quadrant(row):
    cap_ok = row['capacity_achieved'] >= CAP_STD
    qual_ok = row['qualified_rate'] >= QUAL_STD
    if cap_ok and qual_ok:
        return '双优'
    if cap_ok and not qual_ok:
        return '高产低质'
    if not cap_ok and qual_ok:
        return '低产高质'
    return '双低'


n_before = len(ws)
ws = ws[['capacity_achieved', 'qualified_rate']]     .dropna(subset=['capacity_achieved', 'qualified_rate']).copy()
if len(ws) < n_before:
    print(f'⚠️ 提示：{n_before - len(ws)} 个车间因指标缺失被剔除')

# —— 距标准差距（百分点 pp，正数=达标，负数=缺口）——
ws['产能距标准'] = (ws['capacity_achieved'] - CAP_STD).round(2)
ws['良率距标准'] = (ws['qualified_rate'] - QUAL_STD).round(2)
# —— 缺口量（pp）与改善优先级（1=最紧急；只统计未达标方向）——
ws['产能缺口'] = ws['产能距标准'].clip(upper=0).abs()
ws['良率缺口'] = ws['良率距标准'].clip(upper=0).abs()
ws['缺口合计'] = (ws['产能缺口'] + ws['良率缺口']).round(2)
ws['象限'] = ws.apply(classify_quadrant, axis=1)
ws['改善优先级'] = ws['缺口合计'].rank(method='min', ascending=False).astype(int)
ws = ws.sort_values('改善优先级')
# 复制一份，不改动原始df
ws_show = ws.copy()

# 修改列名为中文
ws_show.rename(columns={
    "workshop_name": "车间名称",
    "capacity_achieved": "产能达成率%",
    "qualified_rate": "良品率%",
}, inplace=True)
print('四象限分类结果（改善优先级 1 = 最紧急）：')
# 再输出markdown（Markdown）表格
print(ws_show.to_markdown(index=True, numalign="right", stralign="left"))
print('\n各象限分布：')
for q in ['双优', '高产低质', '低产高质', '双低']:
    members = ws[ws['象限'] == q].index.tolist()
    if members:
        print(f'  {q}（{len(members)} 个）：{"、".join(members)}')
# —— 关键洞察（随数据自动生成）——
n_cap_low = int((ws['产能距标准'] < 0).sum())
n_qual_low = int((ws['良率距标准'] < 0).sum())
print('\n💡 关键洞察：')
if n_qual_low == len(ws):
    print(f'  ① 全部 {len(ws)} 个车间良品率均低于 {QUAL_STD:.0f}% 标准'
          f'（公司整体均值仅 {overall_qual:.2f}%），质量缺口是全局共性短板，'
          f'建议优先排查系统性质量根因（设备/工艺/物料）；')
elif n_qual_low > 0:
    print(f'  ① 有 {n_qual_low} 个车间良品率低于 {QUAL_STD:.0f}% 标准'
          f'（公司整体均值 {overall_qual:.2f}%）；')
else:
    print(f'  ① 全部车间良品率达标（≥ {QUAL_STD:.0f}%），质量水平稳定；')
if n_cap_low > 0:
    worst_cap = ws[ws['产能距标准'] < 0].sort_values('产能距标准').index[0]
    print(f'  ② 产能未达标车间 {n_cap_low} 个，距标准最远的是「{worst_cap}」'
          f'（缺口 {abs(ws.loc[worst_cap, "产能距标准"]):.2f}pp）；')
else:
    print(f'  ② 全部车间产能达成率达标（≥ {CAP_STD:.0f}%）；')
top1 = ws.sort_values('缺口合计', ascending=False).index[0]
print(f'  ③ 当前最需改善：「{top1}」（缺口合计 {ws.loc[top1, "缺口合计"]:.2f}pp'
      f'，其中产能 {ws.loc[top1, "产能缺口"]:.2f}pp、良率 {ws.loc[top1, "良率缺口"]:.2f}pp）。')


四象限分类结果（改善优先级 1 = 最紧急）：
| workshop_name   |   产能达成率% |   良品率% |   产能距标准 |   良率距标准 |   产能缺口 |   良率缺口 |   缺口合计 | 象限     |   改善优先级 |
|:----------------|--------------:|----------:|-------------:|-------------:|-----------:|-----------:|-----------:|:---------|-------------:|
| 机加工车间      |         79.74 |     92.34 |        -0.26 |        -2.66 |       0.26 |       2.66 |       2.92 | 双低     |            1 |
| 冲压车间        |         79.82 |     92.29 |        -0.18 |        -2.71 |       0.18 |       2.71 |       2.89 | 双低     |            2 |
| 焊接车间        |         80.15 |     92.25 |         0.15 |        -2.75 |          0 |       2.75 |       2.75 | 高产低质 |            3 |
| 涂装车间        |         79.81 |     92.45 |        -0.19 |        -2.55 |       0.19 |       2.55 |       2.74 | 双低     |            4 |
| 装配车间        |         80.32 |     92.44 |         0.32 |        -2.56 |          0 |       2.56 |       2.56 | 高产低质 |            5 |
| 质检中心        |         80.46 |     92.47 |    

In [ ]:
# ============================================================
# 第 5 段：可视化 —— 四象限定位图 + 距标准缺口条形图
# ============================================================
import numpy as np
import matplotlib.patches as mpatches

CAP_STD, QUAL_STD = 80.0, 95.0
QCOLOR = {'双优': '#2E9E5B', '高产低质': '#E8A23D',
          '低产高质': '#4C8BF5', '双低': '#E05A5A'}

pts = ws.rename_axis('workshop_name').reset_index()     .sort_values('capacity_achieved').reset_index(drop=True)
counts = ws['象限'].value_counts()

# ================= 画布与四象限主图 =================
# 【动态坐标范围】不再写死：按实际数据 + 标准线自动留边，
# 保证任何车间（哪怕产能低于 70%）都在图上，不会被裁掉。
xmin = min(pts['capacity_achieved'].min(), CAP_STD) - 4.0
xmax = max(pts['capacity_achieved'].max(), CAP_STD) + 4.0
ymin = min(pts['qualified_rate'].min(), QUAL_STD) - 1.5
ymax = max(pts['qualified_rate'].max(), QUAL_STD) + 1.5

fig = plt.figure(figsize=(15.5, 7.6))
gs = fig.add_gridspec(2, 2, width_ratios=[1.55, 1.0], height_ratios=[1, 1],
                      left=0.065, right=0.975, top=0.855, bottom=0.095,
                      wspace=0.34, hspace=0.5)
ax1 = fig.add_subplot(gs[:, 0])   # 四象限主图
ax2 = fig.add_subplot(gs[0, 1])   # 产能距标准
ax3 = fig.add_subplot(gs[1, 1])   # 良率距标准

# 四个象限的淡色底衬
for (x0, x1, y0, y1), q in [
    ((CAP_STD, xmax, QUAL_STD, ymax), '双优'),
    ((CAP_STD, xmax, ymin, QUAL_STD), '高产低质'),
    ((xmin, CAP_STD, QUAL_STD, ymax), '低产高质'),
    ((xmin, CAP_STD, ymin, QUAL_STD), '双低'),
]:
    ax1.add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                     facecolor=QCOLOR[q], alpha=0.09,
                                     edgecolor='none', zorder=0))

# 每个点向 x 轴 / y 轴作虚线投影，便于直接读数
for _, r in pts.iterrows():
    px, py = r['capacity_achieved'], r['qualified_rate']
    ax1.plot([px, px], [ymin, py], color='#C4C4C4', linestyle='--',
             linewidth=0.8, zorder=1)
    ax1.plot([xmin, px], [py, py], color='#C4C4C4', linestyle='--',
             linewidth=0.8, zorder=1)

ax1.set_xlim(xmin, xmax)
ax1.set_ylim(ymin, ymax)
ax1.grid(True, color='#E6E6E6', linewidth=0.8)
ax1.set_axisbelow(True)
for sp in ('top', 'right'):
    ax1.spines[sp].set_visible(False)
for sp in ('left', 'bottom'):
    ax1.spines[sp].set_color('#BBBBBB')

# 管理标准参考线 + 标注
ax1.axvline(CAP_STD, color='#8A8A8A', linestyle='--', linewidth=1.2, zorder=2)
ax1.axhline(QUAL_STD, color='#8A8A8A', linestyle='--', linewidth=1.2, zorder=2)
ax1.text(CAP_STD + 0.35, ymax - 0.28, '产能标准 80%', fontsize=9.5,
         color='#7A7A7A', va='top', ha='left', zorder=3)
ax1.text(xmin + 0.35, QUAL_STD - 0.25, '良品率标准 95%', fontsize=9.5,
         color='#7A7A7A', va='top', ha='left', zorder=3)

# 象限名 + 数量（贴四角，远离数据簇）
zone_pos = {
    '低产高质': (xmin + 0.3, ymax - 0.18, 'left', 'top'),
    '双优':     (xmax - 0.3, ymax - 0.18, 'right', 'top'),
    '双低':     (xmin + 0.3, ymin + 0.18, 'left', 'bottom'),
    '高产低质': (xmax - 0.3, ymin + 0.18, 'right', 'bottom'),
}
for q, (zx, zy, ha, va) in zone_pos.items():
    n = int(counts.get(q, 0))
    ax1.text(zx, zy, f'{q}区 · {n} 个', fontsize=11, color=QCOLOR[q],
             fontweight='bold', ha=ha, va=va, alpha=0.9, zorder=3)

# 散点：外层淡色光环（体现归属象限）+ 内层实心核（区分个体）
for _, r in pts.iterrows():
    ax1.scatter(r['capacity_achieved'], r['qualified_rate'], s=560,
                color=QCOLOR[r['象限']], alpha=0.16, edgecolor='none', zorder=4)
for _, r in pts.iterrows():
    ax1.scatter(r['capacity_achieved'], r['qualified_rate'], s=105,
                color=QCOLOR[r['象限']], edgecolor='white', linewidth=1.5, zorder=5)

# 车间名 +「产能 / 良率」两个比率：统一放点右侧，最右点放左侧
i_right = pts['capacity_achieved'].idxmax()
for i, r in pts.iterrows():
    off, ha = ((-14, 0), 'right') if i == i_right else ((14, 0), 'left')
    txt = f"{r['workshop_name']}（{r['capacity_achieved']:.2f} / {r['qualified_rate']:.2f}）"
    ax1.annotate(txt,
                 xy=(r['capacity_achieved'], r['qualified_rate']),
                 xytext=off, textcoords='offset points',
                 fontsize=10.5, fontweight='bold', color='#2B2B2B', zorder=6,
                 ha=ha, va='center',
                 arrowprops=dict(arrowstyle='-', color='#C2C2C2', lw=0.9,
                                 shrinkA=0, shrinkB=8))

ax1.set_xlabel('产能达成率（%）', fontsize=12.5, labelpad=7)
ax1.set_ylabel('良品率（%）', fontsize=12.5, labelpad=7)
ax1.tick_params(labelsize=10.5)

# ================= 右侧：距标准缺口（pp） =================
def gap_panel(ax, sort_col, title):
    d = pts.sort_values(sort_col, ascending=True)
    vals = d[sort_col].values.astype(float)
    ax.barh(d['workshop_name'].values, vals,
            color=['#2E9E5B' if v >= 0 else '#E05A5A' for v in vals],
            height=0.6, edgecolor='white', linewidth=0.8, zorder=3)
    ax.axvline(0, color='#555555', linewidth=1.1, zorder=4)
    ax.set_title(title, fontsize=12, loc='left', pad=10, color='#2B2B2B')
    ax.grid(axis='x', color='#EAEAEA', linewidth=0.8)
    ax.set_axisbelow(True)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'):
        ax.spines[sp].set_color('#BBBBBB')
    ax.tick_params(labelsize=10.5)
    vmax = max(abs(vals).max(), 1.0)
    ax.set_xlim(-vmax * 1.32, vmax * 1.32)      # 预留数值标签空间，避免被裁
    for b, v in zip(ax.patches, vals):
        ax.annotate(f'{v:+.2f}', xy=(b.get_width(), b.get_y() + b.get_height() / 2),
                    xytext=(5 if v >= 0 else -5, 0), textcoords='offset points',
                    ha='left' if v >= 0 else 'right', va='center',
                    fontsize=10, color='#3A3A3A', fontweight='bold')

gap_panel(ax2, '产能距标准', '产能达成率 - 80% 标准（pp）')
gap_panel(ax3, '良率距标准', '良品率 - 95% 标准（pp）')

fig.suptitle('各车间「产能 × 良率」四象限定位分析', fontsize=17, fontweight='bold', y=0.975)
fig.text(0.5, 0.912,
         '固定管理标准：产能 80% · 良品率 95%　|　点标注括号内为「产能 / 良率」（%）；'
         '右侧正数 = 达标，负数 = 距标准缺口（pp）',
         ha='center', fontsize=10.5, color='#6E6E6E')

plt.savefig('bi/analysis/capacity_quality_quadrant.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('✅ 图表已保存：bi/analysis/capacity_quality_quadrant.png')


In [ ]:
print("\n" + "="*60)
print("📋 改进建议")
print("="*60)

# ⚠️ 修正记录：ws['象限'] 的取值是 '双优' / '高产低质' / '低产高质' / '双低'
#    （见第 4 段的 classify_quadrant，不带 emoji）。
#    原实现拿 '⚡ 高产低质' 这类带 emoji 的字符串去比较，一个分支都匹配不上，
#    结果全部落到 else，把 6 个车间统统打印成「✅ 均达标」——建议完全失真。
ICON = {'双优': '✅', '高产低质': '🔧', '低产高质': '📈', '双低': '🚨'}
ADVICE = {
    '双优':     '产能和良率均达标，保持当前水平',
    '高产低质': '产能达标但良率偏低 → 建议排查质量问题',
    '低产高质': '良率高但产能不足 → 建议提升产能',
    '双低':     '产能和良率均需改进 → 建议全面排查',
}
for idx, row in ws.iterrows():
    q = row['象限']
    print(f"{ICON.get(q, '•')} {idx}（{q}｜产能 {row['capacity_achieved']:.2f}% · "
          f"良率 {row['qualified_rate']:.2f}%）: {ADVICE.get(q, '未分类')}")
print("\n（缺口明细见图表右侧面板，或看第 4 段的表格）")
